# CreditGuard: Phase 2 - Leakage-Safe Feature Engineering Pipeline

This notebook establishes the production-grade, scikit-learn compatible feature engineering and preprocessing pipeline.

### Core Architecture Principles:
1. **Zero Data Leakage**: The dataset is partitioned into train/test splits before computing any reference statistics or transformations.
2. **Corrected Cash Flow Metric**: Replaces the naive `disposable_income = income - loan_amnt` with `net_annual_income_after_debt`, which subtracts the annual amortized loan service from annual income.
3. **Audited Debt Burden Feature**: Implements `high_debt_burden_flag` corresponding to the audited 30% debt-to-income default risk cliff.
4. **Unified ColumnTransformer**: Packages median numerical imputation, scaling, and categorical one-hot encoding.

In [ ]:
import sys
from pathlib import Path
project_root = Path.cwd().parent
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

import pandas as pd
import numpy as np
from src.config import RAW_DATA_PATH, TARGET_COL
from src.ingestion.loader import CreditDataLoader
from src.models.trainer import ModelTrainer
from src.preprocessing.ml_pipeline import (
    CreditRiskFeatureEngineer,
    build_column_preprocessor,
    build_ml_pipeline,
    get_feature_groups
)

print('Feature engineering pipeline modules successfully imported.')

## 1. Stratified Train / Test Isolation
We split the dataset into 80% train and 20% test before calculating any medians, scalers, or encodings.

In [ ]:
loader = CreditDataLoader(RAW_DATA_PATH)
raw_df = loader.load_raw_data().drop_duplicates()
clean_df = raw_df[(raw_df['person_age'] >= 18) & (raw_df['person_age'] <= 100)].copy()
emp_mask = ((clean_df['person_emp_length'] <= 60) & (clean_df['person_emp_length'] <= (clean_df['person_age'] - 14))) | clean_df['person_emp_length'].isna()
clean_df = clean_df[emp_mask].reset_index(drop=True)

trainer = ModelTrainer()
X_train, X_test, y_train, y_test = trainer.split_data(clean_df, target_col=TARGET_COL)
print(f'Training split: {X_train.shape[0]:,} rows | Test split: {X_test.shape[0]:,} rows')
print(f'Train default rate: {y_train.mean():.2%} | Test default rate: {y_test.mean():.2%}')

## 2. Fitting CreditRiskFeatureEngineer on Training Data
The transformer learns group-level imputation medians and calculates the corrected financial metrics.

In [ ]:
fe = CreditRiskFeatureEngineer()
fe.fit(X_train)
print('Grade-level median interest rates learned exclusively from X_train:')
for grade, median_rate in sorted(fe.grade_rate_medians_.items()):
    print(f'  Grade {grade}: {median_rate:.2f}%')

X_train_trans = fe.transform(X_train)
X_test_trans = fe.transform(X_test)

engineered_cols = [
    'income_to_loan_ratio',
    'estimated_annual_debt_service',
    'net_annual_income_after_debt',
    'high_debt_burden_flag',
    'interest_rate_grade_spread'
]
X_train_trans[engineered_cols].head()

## 3. Building ColumnTransformer
Apply numerical median imputation and scaling alongside categorical one-hot encoding.

In [ ]:
preprocessor = build_column_preprocessor(scale_numeric=True)
X_train_proc = preprocessor.fit_transform(X_train_trans)
X_test_proc = preprocessor.transform(X_test_trans)

print(f'Processed train feature matrix shape: {X_train_proc.shape}')
print(f'Processed test feature matrix shape:  {X_test_proc.shape}')
print(f'Nulls remaining in train matrix: {np.isnan(X_train_proc).sum()}')
print(f'Nulls remaining in test matrix:  {np.isnan(X_test_proc).sum()}')